# Problema 1: Sistema de reconocimiento de ESCA (FastAI)

## 1. Configuración inicial y descarga del repositorio

En este notebook trabajamos directamente con el repositorio completo en Google Colab para disponer del dataset y de los archivos de apoyo en una sola descarga.

In [ ]:
# Clonado del repositorio en Colab (si no existe)
import os
from pathlib import Path

repo_url = "https://github.com/kachytronico/PIA_tarea_05_dataset.git"
repo_dir = Path('/content/PIA_tarea_05_dataset')

if not repo_dir.exists():
    !git clone {repo_url} /content/PIA_tarea_05_dataset
else:
    print(f"Repositorio ya presente en: {repo_dir}")

%cd /content/PIA_tarea_05_dataset
!git rev-parse --abbrev-ref HEAD
!ls

## 2. Librerías y detección de rutas del dataset

Instalamos/importamos FastAI y comprobamos automáticamente dónde está la carpeta de imágenes para evitar rutas rígidas.

In [ ]:
# Si FastAI no está disponible en el runtime, descomentar la línea siguiente:
# !pip -q install fastai

from fastai.vision.all import *
from pathlib import Path

repo_root = Path('/content/PIA_tarea_05_dataset')

candidate_paths = [
    repo_root/'esca_dataset'/'esca',
    repo_root/'esca_dataset'/'augmented_esca_dataset',
    repo_root/'esca_dataset'/'healthy',
]

path = next((p for p in candidate_paths if p.exists()), None)
if path is None:
    raise FileNotFoundError(
        'No se encontró una carpeta de dataset válida. Revisa la estructura dentro de esca_dataset/.'
    )

print(f'Ruta detectada para entrenamiento: {path}')

classes = [p.name for p in path.iterdir() if p.is_dir()]
print(f'Clases detectadas ({len(classes)}): {classes}')

n_imgs = len(get_image_files(path))
print(f'Total de imágenes encontradas: {n_imgs}')

## 3. Modelo básico de clasificación (cuadernillo 502)

Construimos un `DataBlock` estándar con división aleatoria reproducible y un `learner` con `resnet18`.